In [1]:
import os, json, time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

with open('../models/prep_config.json') as f:
    cfg = json.load(f)

SEED = cfg['SEED']
label2id = cfg['label2id']
id2label = {v: k for k, v in label2id.items()}
class_weight_id = {label2id[k]: v for k, v in cfg['class_weight'].items()}
target_names = [id2label[i] for i in range(len(id2label))]
np.random.seed(SEED)

In [2]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
train_df['clean'] = train_df['clean'].fillna('')
val_df['clean'] = val_df['clean'].fillna('')

y_train = train_df['sentiment'].map(label2id).values
y_val = val_df['sentiment'].map(label2id).values

if os.path.exists('../models/tfidf_vectorizer.joblib'):
    tfidf = joblib.load('../models/tfidf_vectorizer.joblib')
    X_train_tfidf = tfidf.transform(train_df['clean'])
else:
    tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=20000, min_df=2, sublinear_tf=True)
    X_train_tfidf = tfidf.fit_transform(train_df['clean'])
X_val_tfidf = tfidf.transform(val_df['clean'])

num_cols = ['contentLength', 'n_excl']
X_train_num = csr_matrix(train_df[num_cols].values.astype(np.float32))
X_val_num = csr_matrix(val_df[num_cols].values.astype(np.float32))

X_train_xgb = hstack([X_train_tfidf, X_train_num]).tocsr()
X_val_xgb = hstack([X_val_tfidf, X_val_num]).tocsr()
sample_weight_train = np.array([class_weight_id[y] for y in y_train], dtype=np.float32)

print('X_train_xgb shape:', X_train_xgb.shape)
print('X_val_xgb shape  :', X_val_xgb.shape)

X_train_xgb shape: (15859, 20002)
X_val_xgb shape  : (3398, 20002)


In [ ]:
t0 = time.time()
model_advanced = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=SEED,
    n_jobs=-1
)
model_advanced.fit(X_train_xgb, y_train, sample_weight=sample_weight_train)
train_time = time.time() - t0

t1 = time.time()
y_val_pred = model_advanced.predict(X_val_xgb)
inf_time = time.time() - t1
y_train_pred = model_advanced.predict(X_train_xgb)

print(f'Training time  : {train_time:.4f}s')
print(f'Inference time : {inf_time:.4f}s (3.398 sampel val)')

In [ ]:
metrics_advanced = {
    'model': 'XGBoost (Initial)',
    'train_accuracy': float(accuracy_score(y_train, y_train_pred)),
    'train_macro_f1': float(f1_score(y_train, y_train_pred, average='macro')),
    'val_accuracy': float(accuracy_score(y_val, y_val_pred)),
    'val_macro_precision': float(precision_score(y_val, y_val_pred, average='macro')),
    'val_macro_recall': float(recall_score(y_val, y_val_pred, average='macro')),
    'val_macro_f1': float(f1_score(y_val, y_val_pred, average='macro')),
    'val_weighted_f1': float(f1_score(y_val, y_val_pred, average='weighted')),
    'train_time_sec': float(train_time),
    'val_inf_time_sec': float(inf_time)
}

print('Classification Report — XGBoost Initial (Validation Set):')
print(classification_report(y_val, y_val_pred, target_names=target_names, digits=4))

if os.path.exists('../models/val_metrics_baseline.json'):
    with open('../models/val_metrics_baseline.json') as f:
        metrics_baseline = json.load(f)
    comp_df = pd.DataFrame([metrics_baseline, metrics_advanced])
    display(comp_df[['model', 'train_accuracy', 'val_accuracy', 'val_macro_f1', 'val_weighted_f1', 'train_time_sec']])

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=target_names).plot(ax=ax, cmap='Greens', values_format='d')
ax.set_title('Confusion Matrix — XGBoost Initial (Validation Set)')
plt.tight_layout()
plt.show()

In [ ]:
joblib.dump(model_advanced, '../models/advanced_xgboost_initial.joblib')
with open('../models/val_metrics_advanced_initial.json', 'w') as f:
    json.dump(metrics_advanced, f, indent=2)
print('Saved initial XGBoost model and validation metrics.')